In [21]:
import pandas as pd

In [23]:
df = pd.read_csv('Occupancy_Estimation.csv')

In [ ]:
df.info()





<class 'pandas.DataFrame'>
RangeIndex: 10129 entries, 0 to 10128
Data columns (total 19 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   Date                  10129 non-null  str    
 1   Time                  10129 non-null  str    
 2   S1_Temp               10129 non-null  float64
 3   S2_Temp               10129 non-null  float64
 4   S3_Temp               10129 non-null  float64
 5   S4_Temp               10129 non-null  float64
 6   S1_Light              10129 non-null  int64  
 7   S2_Light              10129 non-null  int64  
 8   S3_Light              10129 non-null  int64  
 9   S4_Light              10129 non-null  int64  
 10  S1_Sound              10129 non-null  float64
 11  S2_Sound              10129 non-null  float64
 12  S3_Sound              10129 non-null  float64
 13  S4_Sound              10129 non-null  float64
 14  S5_CO2                10129 non-null  int64  
 15  S5_CO2_Slope          10129 no

In [25]:
df.head()

,Date,Time,S1_Temp,S2_Temp,S3_Temp,S4_Temp,S1_Light,S2_Light,S3_Light,S4_Light,S1_Sound,S2_Sound,S3_Sound,S4_Sound,S5_CO2,S5_CO2_Slope,S6_PIR,S7_PIR,Room_Occupancy_Count
0,2017/12/22,10:49:41,24.94,24.75,24.56,25.38,121,34,53,40,0.08,0.19,0.06,0.06,390,0.769231,0,0,1
1,2017/12/22,10:50:12,24.94,24.75,24.56,25.44,121,33,53,40,0.93,0.05,0.06,0.06,390,0.646154,0,0,1
2,2017/12/22,10:50:42,25.00,24.75,24.50,25.44,121,34,53,40,0.43,0.11,0.08,0.06,390,0.519231,0,0,1
3,2017/12/22,10:51:13,25.00,24.75,24.56,25.44,121,34,53,40,0.41,0.10,0.10,0.09,390,0.388462,0,0,1
4,2017/12/22,10:51:44,25.00,24.75,24.56,25.44,121,34,54,40,0.18,0.06,0.06,0.06,390,0.253846,0,0,1


In [26]:
# As the date and time were string data types, we need to convert them to datetime data type.
df['Datetime'] = pd.to_datetime(df['Date'] + ' ' + df['Time'], format='%Y/%m/%d %H:%M:%S')

In [27]:
# Now that we have created a new column 'Datetime', we can drop the original 'Date' and 'Time' columns from the dataframe.
df= df.drop(columns=['Date', 'Time'])

In [28]:
df.info("Datetime")


<class 'pandas.DataFrame'>
RangeIndex: 10129 entries, 0 to 10128
Data columns (total 18 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   S1_Temp               10129 non-null  float64       
 1   S2_Temp               10129 non-null  float64       
 2   S3_Temp               10129 non-null  float64       
 3   S4_Temp               10129 non-null  float64       
 4   S1_Light              10129 non-null  int64         
 5   S2_Light              10129 non-null  int64         
 6   S3_Light              10129 non-null  int64         
 7   S4_Light              10129 non-null  int64         
 8   S1_Sound              10129 non-null  float64       
 9   S2_Sound              10129 non-null  float64       
 10  S3_Sound              10129 non-null  float64       
 11  S4_Sound              10129 non-null  float64       
 12  S5_CO2                10129 non-null  int64         
 13  S5_CO2_Slope          10129

In [41]:
pd.DataFrame({'Date': sorted(df['Datetime'].dt.date.unique())})

,Date
0,2017-12-22
1,2017-12-23
2,2017-12-24
3,2017-12-25
4,2017-12-26
5,2018-01-10
6,2018-01-11


- Dec 22 was Friday
- Dec 23 was Saturday
- Dec 24 was Sunday---- no one
- Dec 25 was Monday---- no one
- Dec 26 was Tuesday---- no one
- Jan 10 was Wednesday
- Jan 11 was Thursday---- no one

In [29]:
df['Room_Occupancy_Count'].value_counts()

Room_Occupancy_Count
0    8228
2     748
3     694
1     459
Name: count, dtype: int64

In [35]:
# counting how many occupied sanpshots happen on each day

df[df['Room_Occupancy_Count']>0] ['Datetime'].dt.date.value_counts().sort_index()


Datetime
2017-12-22    825
2017-12-23    782
2018-01-10    294
Name: count, dtype: int64

### Now lets creat a simple yes/no column called occupied. in the data set we have head counts, but i need to kno when was the room occupied and how did the sensors act.
### 1 if occupied, and 0 if empty
## below table has the mean diff of each sensor 

In [55]:
df['occupied']= (df['Room_Occupancy_Count']> 0).astype(int)
df.groupby('occupied').mean(numeric_only=True).round(2).T

occupied,0,1
S1_Temp,25.34,25.96
S2_Temp,25.37,26.30
S3_Temp,24.93,25.61
S4_Temp,25.66,26.16
S1_Light,2.69,123.95
S2_Light,3.05,125.41
S3_Light,13.37,124.62
S4_Light,9.23,30.47
S1_Sound,0.08,0.56
S2_Sound,0.05,0.41


### look into snapshots falls on each day

In [44]:
df['Datetime'].dt.date.value_counts().sort_index()

Datetime
2017-12-22    1462
2017-12-23    2779
2017-12-24    1064
2017-12-25    1716
2017-12-26    1063
2018-01-10     997
2018-01-11    1048
Name: count, dtype: int64

In [45]:
train= df[df['Datetime'].dt.date.astype(str) < '2018-01-10']
test= df[df['Datetime'].dt.date.astype(str) >= '2018-01-10']

##### Train Data is of Dec 22,23,24,25, and 26. In which 22 and 23 were movement and rest silent. 
##### Test Data is of Jan 10 and 11; in which 10 was movement and 11 silent. Kept 11th Jan because to know if the model give false alarm on silent days, or it correctly identifies them

In [47]:
print(len(train), len(test))
print(train['occupied'].mean().round(3), test['occupied'].mean().round(3))

8084 2045
0.199 0.144


In [48]:
df= df.sort_values('Datetime')
df['S6_PIR_5min']= df['S6_PIR'].rolling(10, min_periods= 1).max()


In [50]:
df.groupby('occupied')[['S6_PIR', 'S6_PIR_5min']].mean().round(3)

,S6_PIR,S6_PIR_5min
occupied,,
0,0.002,0.017
1,0.470,0.958


#### Giving S& PIR the same 5mins memory

In [ ]:
df['S7_PIR_5min']= df['S7_PIR'].rolling(10,min_periods= 1).max()
df['PIR_any_5min']= df[['S6_PIR_5min', 'S7_PIR_5min']].max(axis=1)

In [ ]:
df.groupby('occupied')[['S6_PIR_5min', 'S7_PIR_5min', 'PIR_any_5min']].mean().round(3)

,S6_PIR_5min,S7_PIR_5min,PIR_any_5min
occupied,,,
0,0.017,0.009,0.017
1,0.958,0.750,0.985


#### re creating train and test data set, because of our engineered data points; dates remain the same

In [ ]:
train= df[df['Datetime'].dt.date.astype(str)< '2018-01-10'].copy()
test= df[df['Datetime'].dt.date.astype(str)>= '2018-01-10'].copy()
#copy because if we have changes later on, there wont be an ambugity of which table to make the changes in

In [58]:
print(train.shape, test.shape)
print([c for c in train.columns if 'PIR_5min' in c or 'PIR_any' in c])

(8084, 22) (2045, 22)
['S6_PIR_5min', 'S7_PIR_5min', 'PIR_any_5min']


#### Base line model - Logistic Regression:
- 1st step: getting the data in shape for the model.
- why we are doing this before the model itself? A PATTERN LEARNER NEEDS 2 THINGS HANDED TO IT (1) THE INPUT CLUES[ALL OUR SENSOR COLUMNS] IN ONE BOX, AND THE KNOWN ANSWERS[OCCUPIED 1/0] IN ANOTHER BOX. IT STUDIES THE CLUES, MAKES A GUESS, CHECKS IT AGAINST THE ANSWER, AND ADJUSTS. 
- SO BEFORE WE EVEN START WE NEED TO SPLIT OUR TABLE IN "CLUES" AND "ANSWERS"

In [59]:
feature_cols = ['S1_Temp','S2_Temp','S3_Temp','S4_Temp',
                'S1_Light','S2_Light','S3_Light','S4_Light',
                'S1_Sound','S2_Sound','S3_Sound','S4_Sound',
                'S5_CO2','S5_CO2_Slope',
                'S6_PIR_5min','S7_PIR_5min']

X_train= train[feature_cols]
y_train= train['occupied']

X_test= test[feature_cols]
y_test= test['occupied']

In [60]:
print(X_train.shape, y_train.shape, X_test.shape, y_test.shape)

(8084, 16) (8084,) (2045, 16) (2045,)


In [61]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

,"max_iter max_iter: int, default=100Maximum number of iterations taken for the solvers to converge.",1000
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation <regularized-logistic-loss>`) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary <random_state>` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is '

In [62]:
from sklearn.metrics import confusion_matrix, classification_report

y_pred = model.predict(X_test)

print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))

[[1751    0]
 [ 184  110]]
              precision    recall  f1-score   support

           0       0.90      1.00      0.95      1751
           1       1.00      0.37      0.54       294

    accuracy                           0.91      2045
   macro avg       0.95      0.69      0.75      2045
weighted avg       0.92      0.91      0.89      2045



#### I'm taking f1 over accuracy because f1 is a blen of precision and recall; and Accuracy would be "got it right" box, so it will only look at how many did we get right. So in this case accuracy would be ~90%. It will ignore that 184 people were left in the dark!

### So our baseline model (logistic regression) has a recall of 0.37 (37%) and recall says out of 100% the model was only able to detect 37% people in the room. mmy main focus will be to improve recall for this project because we cant miss any people who are actually in the room. its fine if we have false alarm(like it gusses that there is someone when in actual there's no one.)

In [63]:
model_weighted = LogisticRegression(max_iter=1000, class_weight='balanced')
model_weighted.fit(X_train, y_train)

,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",'balanced'
,"max_iter max_iter: int, default=100Maximum number of iterations taken for the solvers to converge.",1000
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation <regularized-logistic-loss>`) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary <random_state>` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Defaul

In [64]:
y_pred_weighted = model_weighted.predict(X_test)
print(confusion_matrix(y_test, y_pred_weighted))
print(classification_report(y_test, y_pred_weighted))

[[1751    0]
 [ 184  110]]
              precision    recall  f1-score   support

           0       0.90      1.00      0.95      1751
           1       1.00      0.37      0.54       294

    accuracy                           0.91      2045
   macro avg       0.95      0.69      0.75      2045
weighted avg       0.92      0.91      0.89      2045



In [65]:
y_proba = model_weighted.predict_proba(X_test)[:, 1]

In [74]:
print(pd.cut(y_proba, [0,0.01,0.1,0.3,0.5,0.7,0.9,0.99,1.0], include_lowest=True).value_counts().sort_index())

(-0.001, 0.01]    1935
(0.01, 0.1]          0
(0.1, 0.3]           0
(0.3, 0.5]           0
(0.5, 0.7]           0
(0.7, 0.9]           0
(0.9, 0.99]          0
(0.99, 1.0]        110
Name: count, dtype: int64


In [66]:
print(y_proba[:10])

[2.16664939e-14 3.42243427e-14 2.89337894e-14 4.18401532e-14
 2.95866974e-14 1.27198386e-13 5.00283463e-14 1.15743635e-13
 1.29966404e-13 1.21465989e-13]


new method: checking time since last motion

In [67]:
motion = df[['S6_PIR', 'S7_PIR']].max(axis=1)
groups = (motion == 1).cumsum()
df['time_since_motion'] = groups.groupby(groups).cumcount()

In [68]:
df.groupby('occupied')['time_since_motion'].median()

occupied
0    707.0
1      0.0
Name: time_since_motion, dtype: float64

In [ ]:
feature_cols_v2 = feature_cols + ['time_since_motion']
# ADDED time_since_motion as a feature to the model

train = df[df['Datetime'].dt.date.astype(str) <  '2018-01-10'].copy()
test  = df[df['Datetime'].dt.date.astype(str) >= '2018-01-10'].copy()

X_train, y_train = train[feature_cols_v2], train['occupied']
X_test,  y_test  = test[feature_cols_v2],  test['occupied']

model_v2 = LogisticRegression(max_iter=1000, class_weight='balanced')
model_v2.fit(X_train, y_train)

y_pred_v2 = model_v2.predict(X_test)
print(confusion_matrix(y_test, y_pred_v2))
print(classification_report(y_test, y_pred_v2))

[[1751    0]
 [ 184  110]]
              precision    recall  f1-score   support

           0       0.90      1.00      0.95      1751
           1       1.00      0.37      0.54       294

    accuracy                           0.91      2045
   macro avg       0.95      0.69      0.75      2045
weighted avg       0.92      0.91      0.89      2045



In [75]:
print(feature_cols_nolight)


['S1_Temp', 'S2_Temp', 'S3_Temp', 'S4_Temp', 'S1_Sound', 'S2_Sound', 'S3_Sound', 'S4_Sound', 'S5_CO2', 'S5_CO2_Slope', 'S6_PIR_5min', 'S7_PIR_5min', 'time_since_motion']


In [70]:
light_cols = ['S1_Light', 'S2_Light', 'S3_Light', 'S4_Light']
feature_cols_nolight = [c for c in feature_cols_v2 if c not in light_cols]

X_train_nl, X_test_nl = train[feature_cols_nolight], test[feature_cols_nolight]

model_nolight = LogisticRegression(max_iter=1000, class_weight='balanced')
model_nolight.fit(X_train_nl, y_train)

y_pred_nl = model_nolight.predict(X_test_nl)
print(confusion_matrix(y_test, y_pred_nl))
print(classification_report(y_test, y_pred_nl))

[[1738   13]
 [   0  294]]
              precision    recall  f1-score   support

           0       1.00      0.99      1.00      1751
           1       0.96      1.00      0.98       294

    accuracy                           0.99      2045
   macro avg       0.98      1.00      0.99      2045
weighted avg       0.99      0.99      0.99      2045



In [76]:
# random forest model
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(n_estimators=1000, class_weight='balanced', random_state=42)
rf.fit(X_train_nl, y_train)

y_pred_rf = rf.predict(X_test_nl)
print(confusion_matrix(y_test, y_pred_rf))
print(classification_report(y_test, y_pred_rf))

[[1740   11]
 [   0  294]]
              precision    recall  f1-score   support

           0       1.00      0.99      1.00      1751
           1       0.96      1.00      0.98       294

    accuracy                           0.99      2045
   macro avg       0.98      1.00      0.99      2045
weighted avg       0.99      0.99      0.99      2045



In [72]:
test = test.copy()
test['pred_empty'] = (y_pred_nl == 0).astype(int)

# count consecutive 'empty' predictions
streak = test['pred_empty'].groupby((test['pred_empty'] == 0).cumsum()).cumcount()
test['empty_streak'] = streak * test['pred_empty']

for N in [10, 20, 30]:   # snapshots ≈ 5, 10, 15 minutes
    release = test['empty_streak'] >= N
    false_release = (release & (test['occupied'] == 1)).sum()
    correct_release = (release & (test['occupied'] == 0)).sum()
    print(f"N={N} (~{N//2} min): false releases={false_release}, correct releases={correct_release}")

N=10 (~5 min): false releases=0, correct releases=1729
N=20 (~10 min): false releases=0, correct releases=1719
N=30 (~15 min): false releases=0, correct releases=1709


In [73]:
import json

results = {
    "project": "Privacy-preserving room occupancy estimation",
    "data": {"train_days": "2017-12-22 to 2017-12-26", "test_days": "2018-01-10 to 2018-01-11",
             "test_snapshots": int(len(test)), "test_occupied_share": round(float(y_test.mean()), 3)},
    "model": {"type": "LogisticRegression", "class_weight": "balanced",
              "features_excluded": light_cols, "reason_excluded": "light dependence caused 184 misses when occupants worked with lights off"},
    "test_performance": {"recall_occupied": 1.00, "precision_occupied": 0.96, "f1_occupied": 0.98,
                         "false_negatives": 0, "false_positives": 13,
                         "false_positive_cause": "CO2 lag ~7 min after departure"},
    "policy": {"rule": "release room after N consecutive predicted-empty snapshots",
               "recommended_N": 20, "recommended_minutes": 10,
               "false_releases_on_test": 0, "correct_release_snapshots": 1719},
    "limitations": ["single occupied test day", "backtest not live experiment", "long-still-occupant case untested",
                    "no HVAC during data collection"],
    "recommendation": "PILOT with human review; do not auto-release without confirmation"
}

with open('occupancy_decision_packet.json', 'w') as f:
    json.dump(results, f, indent=2)
print(json.dumps(results, indent=2))

{
  "project": "Privacy-preserving room occupancy estimation",
  "data": {
    "train_days": "2017-12-22 to 2017-12-26",
    "test_days": "2018-01-10 to 2018-01-11",
    "test_snapshots": 2045,
    "test_occupied_share": 0.144
  },
  "model": {
    "type": "LogisticRegression",
    "class_weight": "balanced",
    "features_excluded": [
      "S1_Light",
      "S2_Light",
      "S3_Light",
      "S4_Light"
    ],
    "reason_excluded": "light dependence caused 184 misses when occupants worked with lights off"
  },
  "test_performance": {
    "recall_occupied": 1.0,
    "precision_occupied": 0.96,
    "f1_occupied": 0.98,
    "false_negatives": 0,
    "false_positives": 13,
    "false_positive_cause": "CO2 lag ~7 min after departure"
  },
  "policy": {
    "rule": "release room after N consecutive predicted-empty snapshots",
    "recommended_N": 20,
    "recommended_minutes": 10,
    "false_releases_on_test": 0,
    "correct_release_snapshots": 1719
  },
  "limitations": [
    "single oc